In [0]:
%run ../config/config

In [0]:
import pandas as pd
import json
import os
import sys
import time
from pyspark.sql.functions import col

sys.path.insert(0, str(PROJECT_PATH))
from utils.certification import certify_scoring
from utils.certification_report_generator import CertificationReportGenerator
from utils.logger import MLOpsLogger, log_dataframe_info

# spark = SparkSession.builder.getOrCreate()

In [0]:
# Logger certification
logger = MLOpsLogger(
    name='07_certification',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '07_certification'
    }
)

In [0]:
# Certification thresholds
certification_thresholds = CERTIFICATION_THRESHOLDS
TABLE_TOLERANCE_PCT = 1.0
TOLERANCE = 0.05
logger.info(f"Configuración de certificación cargada dinámicamente {COMPARE_COLUMNS}")

In [0]:
try:
    logger.info("=" * 60)
    logger.info(f"CERTIFICACIÓN - {ENV.upper()} - Mes {MES_VTA}")
    logger.info("=" * 60)

    start_time = time.time()

    # =============================================================================
    # 1. DETERMINAR AMBIENTES A COMPARAR
    # =============================================================================
    # dev: compara Python (OUTPUT_TABLE) vs SAS (SCORING_TABLE_SAS)
    # stg: compara STG vs DEV
    # prod: compara PROD vs STG

    if ENV.lower() == 'dev':
        env_source = 'dev'
        env_target = 'sas'
        scoring_path_source = OUTPUT_TABLE  # tabla generada por 05_inference
        scoring_path_target = SCORING_TABLE_SAS
    elif ENV.lower() == 'stg':
        env_source = 'stg'
        env_target = 'dev'
        scoring_path_source = OUTPUT_TABLE
        scoring_path_target = OUTPUT_TABLE.replace(f"{ENV}", "dev")
    elif ENV.lower() == 'prod':
        env_source = 'prod'
        env_target = 'stg'
        scoring_path_source = OUTPUT_TABLE
        scoring_path_target = OUTPUT_TABLE.replace(f"{ENV}", "stg")
    else:
        raise ValueError(f"Ambiente no válido: {ENV}. Use 'dev', 'stg' o 'prod'")

    logger.info(f"Comparación: {env_source.upper()} vs {env_target.upper()}")
    logger.info(f"  Source: {scoring_path_source}")
    logger.info(f"  Target: {scoring_path_target}")

    # =============================================================================
    # 2. CARGAR DATOS
    # =============================================================================
    
    logger.info(f"\nCargando datos para codmes={MES_VTA}...")

    df_source = spark.read.format("delta").table(scoring_path_source) \
        .filter(col("codmes") == MES_VTA)
    count_source = df_source.count()
    logger.info(f" {env_source.upper()}: {count_source:,} registros")

    df_target = spark.read.format("delta").table(scoring_path_target) \
        .filter(col("codmes") == MES_VTA) \
        .withColumnRenamed("PD_NUEVO", "numpd")\
        .withColumnRenamed("XB_NUEVO", "NUMXB")\
        .withColumnRenamed("SC_NUEVO", "numscore")\
        .drop("PD_NUEVO_CAL", "XB_NUEVO_CAL")
    count_target = df_target.count()
    logger.info(f" {env_target.upper()}: {count_target:,} registros")

    if count_source == 0:
        raise ValueError(f"No hay datos en {env_source.upper()} para mes_vta={MES_VTA}")

    if count_target == 0:
        logger.warning(f"⚠️ No hay datos en {env_target.upper()} para mes_vta={MES_VTA}")

    # =============================================================================
    # 3. EJECUTAR CERTIFICACIÓN
    # =============================================================================
    logger.info(f"\nEjecutando certificación...")

    certification_thresholds = CERTIFICATION_THRESHOLDS

    cert_report = certify_scoring(
        df_scoring_source=df_source,
        df_scoring_target=df_target,
        env_source=env_source,
        env_target=env_target,
        join_keys=JOIN_KEYS,
        compare_columns=COMPARE_COLUMNS,
        critical_fields=CRITICAL_FIELDS,
        exclude_columns=EXCLUDE_COLUMNS,
        thresholds=certification_thresholds,
        tolerance=TOLERANCE,
        table_tolerance_pct=TABLE_TOLERANCE_PCT if 'TABLE_TOLERANCE_PCT' in locals() else 1.0
    )

    results = cert_report.get_results()
    summary = results['summary']

    logger.info(f"\n{'=' * 60}")
    logger.info("RESULTADOS")
    logger.info(f"{'=' * 60}")
    logger.info(f"Estado: {summary['overall_status']}")
    logger.info(f"Exitosas: {summary['passed']} | Fallidas: {summary['failed']} | Advertencias: {summary['warnings']}")

    # =============================================================================
    # 4. GENERAR REPORTE HTML
    # =============================================================================
    logger.info(f"\nGenerando reporte HTML...")

    template_path = os.path.join(TEMPLATE_REPORT_PATH, 'certification_report_template.html')
    report_generator = CertificationReportGenerator(template_path, thresholds=certification_thresholds)
    os.makedirs(TEMP_REPORTS_CERTIFICATION_PATH, exist_ok=True)

    report_output_path = os.path.join(
        TEMP_REPORTS_CERTIFICATION_PATH,
        f"certification_{env_source}_vs_{env_target}_{MES_VTA}.html"
    )

    report_generator.generate_html(
        cert_report=cert_report,
        output_path=report_output_path,
        mes_vta=MES_VTA,
        scoring_table=OUTPUT_TABLE.split('.')[-1],
        input_table=TABLE_MASTER.split('.')[-1]
    )

    logger.info(f"✓ Reporte generado: {report_output_path}")

    # =============================================================================
    # 5. GUARDAR RESULTADOS
    # =============================================================================
    total_duration = time.time() - start_time

    certification = summary['overall_status'] == 'SUCCESS'
    dbutils.jobs.taskValues.set(key="certification", value=certification)
    dbutils.jobs.taskValues.set(key="report_path", value=report_output_path)
    dbutils.jobs.taskValues.set(key="comparison", value=f"{env_source}_vs_{env_target}")

    if certification:
        logger.info(f"\nCertificación EXITOSA en {total_duration:.2f}s")
    else:
        logger.warning(f"\nCertificación FALLIDA en {total_duration:.2f}s")
        logger.warning(f"Ver detalles en: {report_output_path}")

    logger.info("=" * 60)

    if not certification:
        raise ValueError(f"Certificación FALLIDA. Ver reporte: {report_output_path}")

except Exception as e:
    logger.error(f"ERROR: {str(e)}")
    certification = False
    dbutils.jobs.taskValues.set(key="certification", value=certification)
    raise

finally:
    if 'logger' in locals():
        logger.close()
    MLOpsLogger.cleanup_job_timestamp()